# lu_astra — maximal 8ФМ Classic policy

**Goal:** the strongest deployable Classic policy, targeting a 60+ point average.

`lu_astra` reuses the proven `lu_sol` torch engine (same simulator, same `RLNetwork`
JSON export) with a wider net (1024/512/512), more PPO epochs, a deeper league and a
head-to-head selector that requires the best checkpoint to beat `lu_sol`.

**Hardware:** Kaggle GPU (T4 x2 / P100). **Budget:** ~12 h overnight.

The driver is resumable: if the session ends, re-run the same cell next session.

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.cuda.empty_cache()

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/leonka123/8FM_AI.git"
REPO_DIR = "/kaggle/working/8FM_AI"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Working in:", os.getcwd())

In [ ]:
CONFIG = {
    "total_updates": 8000,
    "chunk_updates": 200,
    "games_per_update": 512,
    "batch_size": 1024,
    "ppo_epochs": 12,
    "warm_start_games": 12000,
    "bc_epochs": 24,
    "hidden1": 1024,
    "hidden2": 512,
    "action_hidden": 512,
    "league_fraction": 0.75,
    "snapshot_pool_size": 16,
    "selection_opponent": "models/lu_sol.json",
    "head_to_head_games": 60,
    "eval_every": 50,
    "eval_games": 80,
    "device": "auto",
    "seed": 82026,
    "time_budget_seconds": 39600,  # 11 h; leave room to save before Kaggle kills the session
}

for key, value in CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, "kaggle/train_lu_astra.py",
    "--work-dir", "/kaggle/working",
    f"--total-updates={CONFIG['total_updates']}",
    f"--chunk-updates={CONFIG['chunk_updates']}",
    f"--games-per-update={CONFIG['games_per_update']}",
    f"--batch-size={CONFIG['batch_size']}",
    f"--ppo-epochs={CONFIG['ppo_epochs']}",
    f"--warm-start-games={CONFIG['warm_start_games']}",
    f"--bc-epochs={CONFIG['bc_epochs']}",
    f"--hidden1={CONFIG['hidden1']}",
    f"--hidden2={CONFIG['hidden2']}",
    f"--action-hidden={CONFIG['action_hidden']}",
    f"--league-fraction={CONFIG['league_fraction']}",
    f"--snapshot-pool-size={CONFIG['snapshot_pool_size']}",
    f"--selection-opponent={CONFIG['selection_opponent']}",
    f"--head-to-head-games={CONFIG['head_to_head_games']}",
    f"--eval-every={CONFIG['eval_every']}",
    f"--eval-games={CONFIG['eval_games']}",
    f"--device={CONFIG['device']}",
    f"--seed={CONFIG['seed']}",
    f"--time-budget-seconds={CONFIG['time_budget_seconds']}",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=False)

In [ ]:
import os
import numpy as np

from eightfm.baselines import ComboPolicy, GreedyPolicy, SupervisedPolicy
from eightfm.policy import load_policy
from eightfm.rl import RLGreedyPolicy, load_rl
from eightfm.rl_train import evaluate_controlled_seat

net = load_rl("/kaggle/working/lu_astra.json")
policy = RLGreedyPolicy(net)
print(f"Loaded lu_astra: features v{net.feature_version}, "
      f"state={net.W1.shape[0]}, hidden={net.W1.shape[1]}/{net.W2.shape[1]}")

opponents = {
    "greedy": GreedyPolicy(seed=31),
    "combo": ComboPolicy(seed=33),
}
if os.path.exists("models/classic-policy.json"):
    opponents["supervised"] = SupervisedPolicy(load_policy("models/classic-policy.json"), seed=32)
if os.path.exists("models/lu_sol.json"):
    opponents["lu_sol"] = RLGreedyPolicy(load_rl("models/lu_sol.json"))

print("\nEvaluation (30 games per player count):")
for name, opp in opponents.items():
    scores = []
    wins = []
    for players in (3, 4, 5, 6, 7):
        report = evaluate_controlled_seat(players, policy, opp, 30, seed_base=10000 + players * 1000)
        scores.append(report.mean_score)
        wins.append(report.win_rate)
    print(f"  vs {name:12s}: score {np.mean(scores):5.1f} | win {np.mean(wins):.3f}")

In [ ]:
# Six-policy round-robin, 500 games per policy.
!python -X utf8 -m eightfm tournament \
    --participants luna,sol,astra,supervised,greedy,rl_classic \
    --astra /kaggle/working/lu_astra.json \
    --players 3,4,5,6,7 \
    --games 100 \
    --output /kaggle/working/tournament_astra_report.json \
    --seed-base 120000 \
    --seed 0

In [ ]:
# On Kaggle everything under /kaggle/working is saved to the notebook Output tab.
import os

for name in (
    "lu_astra.json",
    "lu_astra-latest.json",
    "lu_astra-metrics.json",
    "driver_state.json",
    "tournament_astra_report.json",
):
    path = os.path.join("/kaggle/working", name)
    print(("OK  " if os.path.exists(path) else "MISS") + " " + path)